## Middleware

- These are basically hooks which can act as a trigger for a tool call or a function.
- Can be used to control what happens inside an agent.
- The hooks can be retries, fall backs, rate limits, tranforming prompts, tool selection and output formatting etc.


In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

There are few in-built hooks too like 
- Summarization
- Human In Feedback
... more in Langchain

### Summarization
automatically summarize when approaching token limits

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage


agent = create_agent(
    model="groq:qwen/qwen3.6-27b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:groq/compound-mini",  # We can use flash or mini models for summarization to save tokens
            trigger=("messages", 10),  # At every 10 messages summarization will work.
            keep=("messages", 4),  # To have enough context keep last 4 chats
        )
    ],
)

In [3]:
# What is thread?


config = {"configurable": {"thread_id": "testing-01"}}

In [4]:
import re

from IPython.display import Markdown, display


def strip_thinking(text: str) -> str:
    """qwen wraps reasoning in <think>...</think>, which Markdown would swallow as HTML."""
    return re.sub(r"<think>.*?</think>", "", text, flags=re.S).strip()


questions_list = [
    "In one sentence, what is a vector database?",
    "Name two popular ones.",
    "Which one is easiest to run locally?",
    "How is it different from a normal SQL database?",
    "What is an embedding?",
    "Which model would you use to create embeddings?",
    "What does cosine similarity measure?",
    "Now, what was the very first thing I asked you about?",
]


for q in questions_list:
    result = agent.invoke({"messages": [HumanMessage(content=q)]}, config)
    display(Markdown(f"**Q:** {q}"))
    display(Markdown(strip_thinking(result["messages"][-1].content)))
    print(f"Message count in state: {len(result['messages'])}\n")

**Q:** In one sentence, what is a vector database?

A vector database is a specialized storage system optimized for indexing and retrieving high-dimensional numerical embeddings using similarity-based algorithms instead of traditional exact-match or keyword queries.

Message count in state: 2



**Q:** Name two popular ones.

Two popular vector databases are **Pinecone** and **Milvus**.

...

Cosine similarity measures the **cosine of the angle between two vectors**, which effectively captures how closely aligned their *directions* are, regardless of their length or magnitude.

- **Range:** `-1` to `1` (typically `0` to `1` for normalized AI embeddings)
  - `1` = identical direction (perfect similarity)
  - `0` = orthogonal (no relationship)
  - `-1` = opposite direction (maximum dissimilarity)
- **Why it's used with embeddings:** Semantic meaning is encoded in the *relative proportions* across dimensions, not absolute scale. By ignoring magnitude and focusing purely on orientation, cosine similarity reliably measures how conceptually similar two pieces of data are.
- **In practice:** It's the default metric in most vector databases because it efficiently ranks embeddings by semantic relevance, making it ideal for RAG, recommendation engines, and semantic search.

Message count in state: 8



**Q:** Now, what was the very first thing I asked you about?

Based on the conversation summary, the very first thing you asked about was **vector databases**—specifically requesting a concise definition, examples of popular options, which one is easiest to run locally, and how they differ from a traditional SQL database.

Message count in state: 10

